In [ ]:
import pandas as pd
import sqlite3
import os

# -----------------------------------
# 1. Create / Read E-commerce Data
# -----------------------------------

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "customer_name": ["Rahul", "Priya", "Amit", "Sneha"],
    "city": ["Mumbai", "Pune", "Thane", "Nashik"]
})

products = pd.DataFrame({
    "product_id": [101, 102, 103, 104],
    "product_name": ["Laptop", "Mobile", "Headphones", "Keyboard"],
    "category": ["Electronics", "Electronics", "Accessories", "Accessories"],
    "price": [55000, 25000, 1500, 1200]
})

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004],
    "customer_id": [1, 2, 3, 4],
    "product_id": [101, 102, 103, 104],
    "quantity": [1, 2, 3, 2],
    "order_date": [
        "2026-09-01",
        "2026-09-02",
        "2026-09-03",
        "2026-09-04"
    ]
})

payments = pd.DataFrame({
    "payment_id": [501, 502, 503, 504],
    "order_id": [1001, 1002, 1003, 1004],
    "payment_method": ["UPI", "Card", "Cash", "UPI"],
    "payment_status": ["Completed", "Completed", "Completed", "Pending"]
})


# -----------------------------------
# 2. Data Cleaning
# -----------------------------------

# Remove duplicate records
customers = customers.drop_duplicates()
products = products.drop_duplicates()
orders = orders.drop_duplicates()
payments = payments.drop_duplicates()

# Handle missing values
customers["city"] = customers["city"].fillna("Unknown")
products["price"] = products["price"].fillna(0)

# Convert date into proper format
orders["order_date"] = pd.to_datetime(orders["order_date"])

# Remove invalid prices
products = products[products["price"] > 0]

print("Data Cleaning Completed")


# -----------------------------------
# 3. Transformation
# -----------------------------------

# Merge orders with product information
order_details = orders.merge(
    products,
    on="product_id",
    how="left"
)

# Calculate total amount
order_details["total_amount"] = (
    order_details["quantity"] * order_details["price"]
)

print("\nOrder Details:")
print(order_details)


# -----------------------------------
# 4. Connect to SQL Database
# -----------------------------------

connection = sqlite3.connect("ecommerce.db")

# Load data into SQL tables
customers.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

products.to_sql(
    "products",
    connection,
    if_exists="replace",
    index=False
)

orders.to_sql(
    "orders",
    connection,
    if_exists="replace",
    index=False
)

payments.to_sql(
    "payments",
    connection,
    if_exists="replace",
    index=False
)

order_details.to_sql(
    "order_details",
    connection,
    if_exists="replace",
    index=False
)

print("\nData Loaded Successfully into SQL Database")


# -----------------------------------
# 5. Generate Report
# -----------------------------------

report = pd.read_sql_query("""
SELECT
    p.category,
    SUM(od.quantity) AS total_quantity,
    SUM(od.total_amount) AS total_sales
FROM order_details od
JOIN products p
ON od.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC
""", connection)

print("\n===== SALES REPORT =====")
print(report)


# -----------------------------------
# 6. Historical + New Data
# -----------------------------------

# Newly arriving order
new_order = pd.DataFrame({
    "order_id": [1005],
    "customer_id": [1],
    "product_id": [102],
    "quantity": [1],
    "order_date": [pd.Timestamp("2026-09-05")]
})

# Check whether order already exists
existing_ids = pd.read_sql_query(
    "SELECT order_id FROM orders",
    connection
)

new_order = new_order[
    ~new_order["order_id"].isin(existing_ids["order_id"])
]

# Insert only new records
if not new_order.empty:
    new_order.to_sql(
        "orders",
        connection,
        if_exists="append",
        index=False
    )

print("\nNew Data Added Successfully")

# Close database connection
connection.close()

print("\nETL Pipeline Completed Successfully!")

Data Cleaning Completed

Order Details:
   order_id  customer_id  product_id  quantity order_date product_name  \
0      1001            1         101         1 2026-09-01       Laptop   
1      1002            2         102         2 2026-09-02       Mobile   
2      1003            3         103         3 2026-09-03   Headphones   
3      1004            4         104         2 2026-09-04     Keyboard   

      category  price  total_amount  
0  Electronics  55000         55000  
1  Electronics  25000         50000  
2  Accessories   1500          4500  
3  Accessories   1200          2400  

Data Loaded Successfully into SQL Database

===== SALES REPORT =====
      category  total_quantity  total_sales
0  Electronics               3       105000
1  Accessories               5         6900

New Data Added Successfully

ETL Pipeline Completed Successfully!
